# SPUR code intelligence: graph architecture, worktree overlays, and measured efficiency

**Capture:** 2026-08-21 · SPUR v1.21.0 worktree  
**Artifact:** graph index v4 / schema `spur-graph-schema-v10` · 62,860 nodes · 138,573 resolved edges  
**Scope:** current `spur-graph`, `spur-analyst`, and `code_*` MCP implementation; one live checkout plus a bounded multi-worktree model.

## Executive findings

- The core Parquet query session—open + exact search + symbol read + file manifest + callers—measures **47.6–48.2 ms** on the configured benchmark VM.
- Live MCP calls from this session are **124–369 ms p50** depending on response shape; a four-call exploration is **996 ms p50 / 1,118 ms p95**. The engine is millisecond-scale, while tool transport, metadata, and serialization remain visible.
- On a synthetic 10,000-file repo, a clean warm graph check is **48.1 ms**, a 100-file committed change is **75.3 ms**, and a 10-file dirty change is **67.9 ms**, versus **308.8 ms** cold.
- A new one-file dirty state costs **330.5 ms** to extract and construct. Later exact-code requests with the same fingerprint reuse the extracted artifact and pay only the request-local wrapper/query path: **12.7 ms** to wrap cached artifacts, then **46.5–47.3 ms** for the measured warm multi-operation overlay session. The analyst delta is also persisted on disk for reuse across later sessions.
- Current stores occupy **837.7 MiB** in total: 326.5 MiB shared artifacts, 337.1 MiB active worktree graph, 173.8 MiB DuckDB, and 0.32 MiB overlay cache.
- For 13 linked dirty worktrees, the conservative overlay model uses **850.3 MiB**, versus **6,640.7 MiB** for 13 independent graph+DuckDB copies—**87.2% less disk**.
- Catalog Z3 rules verify the 13-overlay model under a 2 GiB comparison pool and reject the 13-copy model. The exact overlay boundary is 1,210 pass / 1,211 fail.

These are measured and solver-bounded claims, not a promise of identical latency on every host. Core Criterion results and live MCP results include different layers and should not be compared as if they used the same clock boundary.

## 1. Current architecture at a glance

SPUR has two complementary code-intelligence planes:

- **Exact plane (`spur-graph`)** — typed symbols, files, edges, spans, selectors, bounded traversal, temporal identity, and response-level freshness. This plane powers `code_symbol_search`, `code_read_symbol`, `code_callers`, `code_callees`, and `code_subgraph`.
- **Analyst plane (`spur-analyst`)** — DuckDB + retrieval sidecars for concept search, documentation navigation, SQL aggregation, PageRank/communities, paths, temporal scoring, and bounded evidence packs. It orients; exact graph reads ground implementation claims.

The extraction pipeline is deliberately front-loaded. Worktree bytes are parsed once into stable, columnar facts; repeated agent questions then read indexed structures instead of re-parsing or grepping the repository.

```mermaid
flowchart LR
    WT["Git worktree"] --> DISC["ignore-aware discovery"]
    DISC --> TS["tree-sitter extraction<br/>15 languages"]
    TS --> IDS["stable IDs + typed facts"]
    IDS --> PQ["Parquet artifact<br/>ZSTD / 16K row groups"]
    PQ --> EXACT["GraphQueryClient<br/>Parquet · memory · overlay"]
    EXACT --> MCP["code_* MCP tools"]
    PQ --> DB["DuckDB analyst index"]
    PQ --> LANCE["Lance retrieval sidecars"]
    DB --> PACK["knowledge packs / SQL / graph algorithms"]
    LANCE --> PACK
    EDITS["dirty files"] --> DELTA["changed-file delta"]
    DELTA --> EXACT
    DELTA --> DB
```

The architecture sketch is explanatory, not solver-bound. Formal proofs later in the notebook cover the finite storage capacity and overlay state model.

## 2. Technology stack and why each layer exists

| Layer | Current technology | Design reason |
|---|---|---|
| Implementation | Rust 2021; `unsafe_code = "deny"` | Predictable memory/layout, small native binaries, and shared types from extractor to MCP. |
| Parsing | tree-sitter 0.25; 11 grammar crates driving 15 language modes | Incremental, error-tolerant syntax with one normalized symbol/edge schema. |
| Git | `gix` 0.77 plus blob OIDs | Detect changed files by content identity; resolve refs, history, and linked-worktree common dirs. |
| Identity | BLAKE3 + SHA-1/SHA-2 | Stable symbol/file identities, graph content hashes, and Git compatibility. |
| Structural store | Arrow/Parquet, ZSTD-3, 16,384-row groups | Compact columnar persistence and selective scans without a resident database server. |
| Exact query | `GraphQueryClient`; `ParquetClient`, `InMemoryClient`, `OverlayClient`; `petgraph` | One API across persisted, test/in-memory, and dirty-worktree views. |
| Analyst | bundled DuckDB; DuckPGQ/Onager integration | SQL aggregation, FTS/BM25, paths, centrality, communities, and temporal ranking. |
| Retrieval | LanceDB/Lance Index + optional `fastembed` FP32/768 models | Hybrid semantic reranking while retaining BM25 fallback and exact graph grounding. |
| Concurrency | Tokio + `OnceCell`-style singleflight coordinator | Share one delta build among concurrent MCP requests for the same worktree state. |
| Interface | `spur-mcp` tool registry and structured JSON responses | Stable tool contracts, bounded outputs, selectors, and explicit freshness metadata. |

Default analyst limits are 4 GiB per DuckDB connection and 4 threads. The `embed` feature is on by default; disabling it removes the largest storage sidecars but also removes vector reranking.

## 3. Query execution path

An exact query is not “run arbitrary SQL over the whole repository.” The handler resolves a compact selector and dispatches through `GraphQueryClient`:

1. Resolve active worktree and requested project.
2. Load the selected Parquet base, or reuse an already-open client.
3. If tracked files changed, obtain a singleflight overlay keyed by HEAD, base hash, and dirty OID set.
4. Execute name lookup, symbol read, inbound/outbound edge lookup, or bounded traversal.
5. Stamp the response with graph hash, indexed/worktree HEAD, rebuild status, and per-answer file-OID match.

The analyst plane uses DuckDB/Lance to find relevant candidates, then returns graph selectors for exact follow-up. This is why the recommended retrieval stack is **orient → exact symbol → aggregate only when needed**.

```mermaid
sequenceDiagram
    participant A as Agent
    participant M as code_* MCP
    participant R as RebuildCoordinator
    participant B as Base seed
    participant O as OverlayClient
    participant P as ParquetClient
    A->>M: search / read / callers / callees
    M->>R: HEAD + dirty OIDs
    R->>B: self pointer, else main pointer
    alt changed tracked files
        R->>O: build or reuse delta
        O->>P: query immutable base
        O-->>M: shadow + remap + merge edges
    else clean state
        B->>P: open selected artifact
        P-->>M: exact rows
    end
    M-->>A: rows + freshness metadata
```

The sequence is explanatory. The bounded state model in section 9 separately verifies the normalized `NeedBase → SharedBase → DeltaReady → IsolatedAnswer` trace.

## 4. Measurement protocol

Five boundaries are reported separately:

1. **Live MCP round trip.** Fifteen samples after three warmups, timed around actual nested MCP calls with a millisecond clock. This includes server dispatch, response construction, freshness metadata, serialization, and tool transport. p50/p95 use nearest-rank order statistics.
2. **Core query session.** Criterion, optimized bench profile, 100 samples on the configured remote VM. Each iteration opens Parquet, performs exact symbol search, reads the symbol and file manifest, and finds callers. Repository extraction happens once in benchmark setup and is excluded.
3. **Overlay construction.** Criterion with 10 samples, 1 s warmup, and 10 s target measurement per operation. This separates Parquet open, delta extraction, cached wrapping, and full `OverlayClient::new`.
4. **Warm overlay query session.** Criterion optimized locally against the live artifact, 20 samples with 1 s warmup and an 8 s target. The base and one-file `OverlayClient` are constructed before timing, so this measures repeated search + read + manifest + callers against an already-warm view; it deliberately excludes extraction and `from_artifacts` wrapping.
5. **Incremental build.** Criterion quick mode on a synthetic 10,000-file repository with a 100-file committed change and 10 dirty files. The benchmark reports discovery, hashing, and extraction phases.

Storage uses `du -sk` at capture time. Solver models use those exact KiB values. The first Criterion invocation exposed a stale absolute fixture path (`/Volumes/Projects/spur`); the successful run explicitly selected the current artifact under `/Volumes/Projects/Projects/spur`.

Historical committed baselines (`benches/pre_pr2.md`, `baselines.json`) are useful design provenance, but they are not substituted for the fresh results below.

## 5. Latency results

### Live `code_*` MCP path

| Operation | N | p50 ms | p95 ms | Range ms |
|---|---:|---:|---:|---:|
| exact `code_symbol_search` | 15 | **124** | **137** | 118–137 |
| `code_read_symbol` | 15 | **272** | **351** | 264–351 |
| `code_callers` | 15 | **309** | **353** | 304–353 |
| `code_callees` | 15 | **303** | **359** | 294–359 |
| `code_subgraph` radius 1 | 15 | **369** | **403** | 357–403 |
| search + read + callers + callees | 15 | **996** | **1,118** | 983–1,118 |

### Core engine

Criterion `parquet_open_then_session`: **47.593–48.214 ms** estimate interval across 100 samples. This is the best answer to “can the graph support millisecond queries?”: yes, the multi-operation core session is about 48 ms on the benchmark host. The live tool path is higher because it includes layers the core benchmark intentionally excludes.

In [ ]:
xychart-beta
    title Live code tool latency
    x-axis [search, read, callers, callees, subgraph]
    bar [124, 272, 309, 303, 369]
    line [137, 351, 353, 359, 403]

### Incremental and overlay efficiency

#### 10,000-file incremental benchmark

| Scenario | Estimate ms | `ls-files` | BLAKE3 | Extraction |
|---|---:|---:|---:|---:|
| clean cold, 10,000 changed | **308.78** | 23.16 | 1.72 | 258.51 |
| clean warm, cache hit | **48.10** | 22.38 | 1.54 | 0 |
| committed change, 100 files | **75.29** | 23.69 | 1.59 | 20.13 |
| dirty change, 10 files | **67.95** | 21.86 | 1.81 | 17.54 |

A warm check is about **6.4× faster** than cold. Re-extracting 100 of 10,000 files costs about 24% of cold total time. Discovery is now the lower bound for warm builds.

#### Overlay cold build versus warm reuse

| Boundary | Estimate | Paid when |
|---|---:|---|
| Parquet client open | **44.8 µs** | base view opens |
| extract empty delta | **80.3 ms** | extraction is forced with no changed paths |
| extract one-file delta | **320.9 ms** | a new changed-files fingerprint is first seen |
| wrap already-extracted one-file artifacts | **12.7 ms** | an exact-code request creates its request-local `OverlayClient` |
| full one-file `OverlayClient::new` | **330.5 ms** | cold extraction + wrapper construction |
| warm base multi-operation query session | **41.1–41.4 ms** | repeated query over an already-built base client |
| warm one-file overlay query session | **46.5–47.3 ms** | repeated query over an already-built overlay client |

The **330.5 ms is a first-build cost for one new dirty-state key, not a full repository rebuild and not the cost of every query**. On the exact `code_*` path, the extracted delta is cached in process by worktree + changed-files fingerprint. A later request with the same fingerprint skips the 320.9 ms extraction, constructs a request-local wrapper from the cached artifact (12.7 ms in isolation), and queries the warm base+delta view.

Once the overlay client itself already exists, the benchmarked multi-operation session is **46.8 ms** at the midpoint versus **41.2 ms** on the base—about **5.6 ms / 13.6%** overhead. This warm-session number excludes both extraction and wrapper construction, so it must not be substituted for end-to-end MCP latency.

In [ ]:
xychart-beta
    title Incremental graph benchmark 10k files
    x-axis [cold, warm, change_100, dirty_10]
    bar [309, 48, 75, 68]

## 6. Measured storage footprint

The current checkout has **one** Git worktree. The active pointer is a symlink to the worktree-local artifact:

`.spur/graph/CURRENT` → `.spur/graph/1486a9…parquet`

The Git common directory also holds three content-addressed base generations. These stores are related but not identical: the active local artifact includes temporal shards, snapshots, commits, and diagnostics that are absent from the shared base copy.

| Path | Exact KiB | MiB | Role |
|---|---:|---:|---|
| `.git/spur-graph/artifacts/` | 334,356 | **326.52** | Shared immutable base cache; 3 retained generations (24.73, 24.79, 274.79 MiB). |
| `.spur/graph/` | 345,152 | **337.06** | Active worktree artifact selected by `CURRENT`; one generation. |
| `.spur/analyst.duckdb` | 177,932 | **173.76** | Warm SQL/search/graph projection. |
| `.spur/analyst-overlays/` | 328 | **0.32** | Three retained dirty-worktree delta sessions. |
| **Total** | **857,768** | **837.66** | Observed disk across these four stores. |

The active manifest reports 62,860 nodes, 138,573 resolved edges, 158,399 unresolved edges, 2,888 files, 706,948 symbol snapshots, and 734,124 temporal edges. Parquet uses ZSTD level 3 and 16,384-row groups.

## 7. Multi-worktree storage model

The model separates a measured fixed cost from a conservative per-worker delta:

- fixed shared/local stores = 334,356 + 345,152 + 177,932 = **857,440 KiB**
- per dirty linked worktree = **1,024 KiB** (current retained overlay cache is only 328 KiB total)
- comparison pool = **2,097,152 KiB** (2 GiB)

Therefore `857440 + 1024 × W <= 2097152`, so the exact integer boundary is **W = 1,210**. This is a capacity bound for the encoded disk model, not a practical concurrency recommendation.

In [ ]:
flowchart TD
    SPEC["`@spec DESIGNED-DISK
@type Fit = enum[fits, overflow]
@input worktrees: Int
@output status: Fit
@requires BOUNDS: worktrees >= 1 and worktrees <= 2000`"]
    FITS["`@branch FITS
@when worktrees <= 1210
@ensures FITS_STATUS: status = fits`"]
    OVER["`@branch OVER
@when worktrees > 1210
@ensures OVER_STATUS: status = overflow`"]
    CHECK["`@verify CONSIST: witness consistency
@verify DET: prove determinism
@verify COVER: prove partition_coverage
@verify EXCL: prove partition_exclusive
@verify STATUSES: witness each status`"]
    SPEC --> FITS --> CHECK
    SPEC --> OVER --> CHECK

### Why both shared and local graph stores exist

The shared Git-common store is the reusable base cache. The local `.spur/graph` artifact is the worktree-selected, enriched graph and currently includes about 62 MiB of temporal history plus 250 MiB of embedding/search sidecars. A linked worker first tries its own pointer, then resolves the main worktree through `git rev-parse --git-common-dir` and uses that pointer as its seed.

This means “shared” does not imply “all bytes exist once.” The design shares immutable bases and avoids a full graph+DuckDB copy per worker, while allowing one active enriched artifact and tiny dirty deltas. Retention still matters: the shared store currently keeps three generations.

In [ ]:
xychart-beta
    title Retained graph generations MiB
    x-axis [shared_old_1, shared_old_2, shared_current, active_local]
    bar [25, 25, 275, 337]

### Active graph composition (337 MiB)

The current local artifact is dominated by retrieval sidecars, not the structural fact tables:

| Component | KiB | Rounded MiB |
|---|---:|---:|
| `code_symbols.lance` | 163,516 | 160 |
| `sections.lancedb` | 92,448 | 90 |
| Temporal edges + symbol snapshots + commits | 63,400 | 62 |
| Live nodes/edges/manifests/diagnostics and remainder | 25,788 | 25 |

The two embedding/search sidecars account for about **74%** of the active graph. This is the primary storage tradeoff behind hybrid retrieval quality.

In [ ]:
pie title Active graph composition MiB
    "Code symbol Lance sidecar" : 160
    "Section LanceDB sidecar" : 90
    "Temporal history" : 62
    "Live graph facts" : 25
%% @ns-total 337

### Disk by store

The active graph and shared artifact cache are similar in size; DuckDB is about half either graph store, and retained overlay deltas are three orders of magnitude smaller. Values in the chart are rounded MiB; the table above is authoritative.

In [ ]:
xychart-beta
    title Disk by store MiB
    x-axis [shared_artifacts, active_graph, analyst_duckdb, overlays]
    bar [327, 337, 174, 1]

### Thirteen-worktree comparison

Using exact KiB:

- **overlay design:** 857,440 fixed + 13 × 1,024 = **870,752 KiB (850.34 MiB)**
- **independent copies:** 13 × (345,152 graph + 177,932 DuckDB) = **6,800,092 KiB (6,640.71 MiB)**
- **observed one-worktree stores:** **857,768 KiB (837.66 MiB)**

The 13-worktree overlay model uses **87.2% less disk** than independent copies (7.81× smaller). It is a conservative model because the current three-session overlay cache totals only 328 KiB.

In [ ]:
xychart-beta
    title Storage models MiB
    x-axis [overlay_13, independent_13, observed_1]
    bar [850, 6641, 838]

### Capacity boundary under a 2 GiB comparison pool

Fresh catalog solves establish both sides of each boundary:

- shared base + active graph + DuckDB + **1,210** conservative 1 MiB overlays: pass
- the same fixed cost + **1,211** overlays: fail
- **4** independent graph+DuckDB copies: pass
- **5** independent copies: fail

The overlay boundary is about **302×** the independent-copy boundary. This ratio is specific to the measured stores, 1 MiB overlay cap, and 2 GiB pool.

In [ ]:
xychart-beta
    title Max worktrees or overlays on 2048 MiB pool
    x-axis [linked_overlay, independent_copy]
    bar [1210, 4]

### Rounded byte flow

The 337 MiB active graph decomposes into 250 MiB of retrieval sidecars, 62 MiB of temporal history, and 25 MiB of live structural facts. DuckDB is a separate 174 MiB projection for BM25/FTS, scorecards, paths, communities, and temporal queries. Overlay cache is rounded up from 0.32 MiB to 1 MiB in this visualization.

This flow is descriptive. It does not claim that DuckDB bytes are a byte-for-byte subset of Parquet or that sidecars are zero-copy.

In [ ]:
sankey

WorkspaceStores,SharedArtifacts,327
WorkspaceStores,ActiveGraph,337
WorkspaceStores,AnalystDuckDB,174
WorkspaceStores,OverlayCache,1
ActiveGraph,RetrievalSidecars,250
ActiveGraph,TemporalHistory,62
ActiveGraph,LiveGraphFacts,25

## 8. Multi-worktree overlay mechanics

A linked worker does not eagerly clone the full graph:

1. `load_base_seed_for_worktree` tries the worker's own graph pointer; if it is absent, `main_worktree_root` derives the main checkout from Git's common directory and loads that immutable base.
2. The analyst path similarly prefers a local `.spur/analyst.duckdb`, then falls back to the parent database for workers under `.spur/worktrees/<id>`.
3. Changed paths are identified by content OID. Only those paths are re-extracted; the base remains immutable and shared.
4. On the exact `code_*` path, `OverlayClient` shadows base symbols on changed paths, remaps shifted stable IDs, and merges caller/callee edges across the base/delta boundary.
5. On the analyst path, the delta is written under `<worktree>/.spur/analyst-overlays/<cache-key>` and DuckDB opens a base+delta merge session.
6. Responses carry graph hash, indexed/worktree HEAD, rebuild status, and per-response file-OID matching so a consumer can judge freshness.

The two query planes deliberately use different reuse mechanisms. Exact graph requests keep extracted artifacts in process and create a request-local `OverlayClient` around them. Analyst queries retain an in-process merge session **and** persist complete delta directories on disk. That distinction determines what remains warm after a new request versus after a daemon restart.

### Overlay persistence and warmness: four distinct states

“Warm” is not one cache hit. It depends on which layer is being reused:

| State | Reused object | Where it lives | Survives a new request? | Survives daemon restart? |
|---|---|---|---:|---:|
| **cold dirty state** | nothing yet | — | — | — |
| **exact extracted-warm** | changed-file `GraphIndexArtifact` + shadowed paths | process-global LRU, capacity 1 | **yes** | no |
| **analyst session-hot** | `Arc<OverlayMergeSession>` | process memory, capacity 1 | **yes** | no |
| **analyst persisted-warm** | complete delta artifact directory | `<worktree>/.spur/analyst-overlays/<key>`, newest 3 retained | **yes** | **yes** |

#### Exact `code_*` lifecycle

1. A request fingerprints the changed paths for this worktree.
2. On a new fingerprint, extraction builds the changed-file artifact. The measured one-file cold construction is **330.5 ms**, of which **320.9 ms** is extraction.
3. The cache key is `(worktree, changed_files_fingerprint)`. A repeated request with the same key receives the same cached `Arc<GraphIndexArtifact>`; the reuse test confirms the build closure runs once.
4. The current handler then calls `OverlayClient::from_artifacts` for that request. That request-local wrapper costs **12.7 ms** in the isolated benchmark; it is not another extraction and not a full rebuild.
5. Queries merge the immutable base with the cached delta. When the overlay client is already constructed, the measured multi-operation session is **46.5–47.3 ms** versus **41.1–41.4 ms** on the base.

So **yes: the ≈300 ms work is normally first-use only for an unchanged dirty-state fingerprint**. For the exact graph plane, however, the persistent object is the extracted delta in process—not the complete `OverlayClient` across daemon restarts.

#### Analyst lifecycle and disk persistence

The analyst key is stronger: `(HEAD, base_graph_content_hash, dirty_oid_set_hash)`. Its directory name embeds prefixes of HEAD and base hash plus the dirty-set hash. `write_delta_for_session` first reads `manifest.json`; when the manifest is complete and its manifest/graph-index versions match the base, it returns the existing directory without rewriting it. The session coordinator then returns the retained merge session when still in process, or reopens the persisted delta after a later process starts.

The disk cache keeps the three newest delta directories and protects directories held by active sessions. The verified reuse test writes a sentinel into a complete directory and confirms a same-key call leaves it intact.

#### What makes it cold again?

The cost is once **per cache key**, not once forever. A new build/reopen decision occurs when:

- a tracked file's content OID changes, so the changed-files fingerprint or dirty-set hash changes;
- HEAD changes;
- the analyst base graph content hash changes;
- the persisted manifest is missing, incomplete, or version-incompatible;
- the in-process capacity-1 entry is evicted by another state/worktree;
- the exact-code daemon restarts, because its extracted-delta cache is memory-only; or
- an analyst delta ages beyond the three retained disk entries and is pruned.

This is why “first cold, then warm” is correct for a stable worktree state, while edits, rebases, base refreshes, eviction, and some restarts intentionally create a new cold boundary.

In [ ]:
stateDiagram-v2
    [*] --> NeedBase
    NeedBase --> SharedBase: select_base
    SharedBase --> DeltaReady: build_delta
    DeltaReady --> IsolatedAnswer: merge_read
    note right of NeedBase
      @spec WORKER-QUERY
      @type Event = enum[select_base, build_delta, merge_read]
      @input event: Event
      @state-var crosstalk: Bool
      @requires PRE: crosstalk = false
      @state NeedBase
      @invariant NOLEAK: crosstalk = false
    end note
    note right of SharedBase
      @state SharedBase
      @transition SEED
      @from NeedBase
      @to SharedBase
      @event event = select_base
      @guard crosstalk = false
      @update crosstalk' = false
    end note
    note right of DeltaReady
      @state DeltaReady
      @transition DELTA
      @from SharedBase
      @to DeltaReady
      @event event = build_delta
      @guard crosstalk = false
      @update crosstalk' = false
    end note
    note right of IsolatedAnswer
      @state IsolatedAnswer
      @transition MERGE
      @from DeltaReady
      @to IsolatedAnswer
      @event event = merge_read
      @guard crosstalk = false
      @update crosstalk' = false
      @verify INIT: prove initiate NOLEAK
      @verify PSEED: prove preserve NOLEAK on SEED
      @verify PDELTA: prove preserve NOLEAK on DELTA
      @verify PMERGE: prove preserve NOLEAK on MERGE
    end note

## 9. Catalog solver evidence

All storage facts are exact KiB. `sat/pass` validates the supplied model; `unsat/fail` rejects it. A bounded-trace `unsat/infeasible` excludes a witness only from the encoded finite state/event domains and horizon.

### Resource family — `resource.aggregate_capacity`

| Model | Raw status / outcome | Persisted solve |
|---|---|---|
| Fixed 857,440 + 13 × 1,024 KiB vs 2 GiB | **sat / pass** | `sol_01eac9f26e3a47ba` |
| 13 × 523,084 KiB independent copies | **unsat / fail** | `sol_0f072a7e96a74338` |
| Overlay W=1,210 | **sat / pass** | `sol_e14a749777604515` |
| Overlay W=1,211 | **unsat / fail** | `sol_9d22b00567b943e5` |
| Independent W=4 | **sat / pass** | `sol_1c94b5c344d94f6e` |
| Independent W=5 | **unsat / fail** | `sol_77af7c3e3a6e40c3` |

### Workflow family — finite overlay trace

| Query | Outcome | Persisted solve |
|---|---|---|
| NeedBase → SharedBase → DeltaReady → IsolatedAnswer; initial, transitions, safety, reachability | **sat / pass** at horizon 3 | `sol_54a85761fc2f4485` |
| Synthesize a `CrossTalk` target using only those enabled transitions | **unsat / infeasible** at horizon 3 | `sol_a0017565f1314320` |

The workflow result is not an unbounded proof and does not model arbitrary filesystem corruption or a foreign transition omitted from the declared relation.

## 10. Executable summary tables

This Python cell repeats the measured values used by the Mermaid charts: store sizes, live MCP p50/p95 latency, 10k-file incremental timings, overlay construction timings, and capacity boundaries. It is presentation-only; the authoritative sources are the benchmark outputs and manifest inventory documented above.

In [5]:
from IPython.display import HTML, display
import pandas as pd

stores = pd.DataFrame({
    "store": ["shared_artifacts", "active_graph", "analyst_duckdb", "overlay_cache"],
    "kib": [334356, 345152, 177932, 328],
})
stores["mib"] = (stores["kib"] / 1024).round(2)

live_mcp = pd.DataFrame({
    "operation": ["symbol_search", "read_symbol", "callers", "callees", "subgraph_r1", "four_call_session"],
    "p50_ms": [124, 272, 309, 303, 369, 996],
    "p95_ms": [137, 351, 353, 359, 403, 1118],
})

incremental = pd.DataFrame({
    "scenario": ["cold_10k", "warm_10k", "change_100", "dirty_10"],
    "estimate_ms": [308.78, 48.10, 75.29, 67.95],
})

overlay = pd.DataFrame({
    "operation": [
        "parquet_open", "extract_empty", "cached_one_file_wrap",
        "extract_one_file", "new_one_file", "warm_base_session", "warm_overlay_session",
    ],
    "estimate_ms": [0.0448, 80.30, 12.734, 320.91, 330.53, 41.244, 46.845],
})

capacity = pd.DataFrame({
    "architecture": ["linked_overlay", "independent_copy"],
    "max_under_2gib": [1210, 4],
})

def html_bars(labels, values, title, color):
    mx = max(values) or 1
    rows = []
    for label, value in zip(labels, values):
        width = max(1, int(round(100 * value / mx)))
        rows.append(
            "<div style='margin:8px 0;font:13px/1.3 ui-sans-serif,system-ui'>"
            f"<div style='display:flex;justify-content:space-between'><span>{label}</span>"
            f"<strong>{value}</strong></div>"
            "<div style='background:#e5e7eb;height:18px;border-radius:4px;overflow:hidden'>"
            f"<div style='width:{width}%;height:18px;background:{color}'></div></div></div>"
        )
    return HTML(f"<div style='margin:1rem 0'><h4>{title}</h4>{''.join(rows)}</div>")

for frame in [stores, live_mcp, incremental, overlay, capacity]:
    display(frame)

display(html_bars(stores["store"], stores["mib"], "Current stores (MiB)", "#2563eb"))
display(html_bars(live_mcp["operation"], live_mcp["p50_ms"], "Live MCP p50 (ms)", "#7c3aed"))
display(html_bars(incremental["scenario"], incremental["estimate_ms"], "Incremental graph time (ms)", "#059669"))
display(html_bars(overlay["operation"], overlay["estimate_ms"], "Overlay boundaries (ms)", "#0f766e"))
display(html_bars(capacity["architecture"], capacity["max_under_2gib"], "2 GiB capacity boundary", "#d97706"))

,store,kib,mib
0,shared_artifacts,334356,326.52
1,active_graph,345152,337.06
2,analyst_duckdb,177932,173.76
3,overlay_cache,328,0.32


,operation,p50_ms,p95_ms
0,symbol_search,124,137
1,read_symbol,272,351
2,callers,309,353
3,callees,303,359
4,subgraph_r1,369,403
5,four_call_session,996,1118


,scenario,estimate_ms
0,cold_10k,308.78
1,warm_10k,48.10
2,change_100,75.29
3,dirty_10,67.95


,operation,estimate_ms
0,parquet_open,0.0448
1,extract_empty,80.3000
2,cached_one_file_wrap,12.7340
3,extract_one_file,320.9100
4,new_one_file,330.5300
5,warm_base_session,41.2440
6,warm_overlay_session,46.8450


,architecture,max_under_2gib
0,linked_overlay,1210
1,independent_copy,4


## 11. Conclusions, caveats, and evidence map

### What is working

- Exact code navigation is backed by a typed, content-addressed graph rather than text matching.
- The core multi-operation query session is below 50 ms on the benchmark VM.
- Incremental discovery dominates warm checks; parsing is zero on a cache hit and limited to changed paths on misses.
- Base+delta overlays avoid rebuilding or duplicating the 337 MiB active graph and 174 MiB analyst DB per worker.
- Stable IDs, path shadowing, edge merging, singleflight, and response freshness metadata make the overlay semantics explicit and inspectable.
- Exact changed-file artifacts are reused in process, while complete analyst deltas are persisted on disk and reopened by later sessions when their key and manifest remain compatible.

### What still costs

- Live MCP round trips are 2.6–7.7× slower than the 48 ms core session for individual operations because the boundaries differ and include orchestration/serialization.
- Retrieval sidecars consume 250 MiB, about 74% of the active artifact.
- The one-file overlay cold path is about 330 ms for each new dirty-state key. A same-key exact request skips extraction but still constructs a request-local wrapper; only the analyst delta survives a daemon restart on disk.
- Three retained shared generations cost 326.5 MiB even with only one current checkout.
- During initial orientation, `knowledge_context_pack_2` reported an analyst hash behind the exact graph hash and skipped graph reasoning. Exact `code_*` reads returned `response_file_oids_match=true`, so implementation claims in this notebook use exact reads.

### Reproduction

- Core session: `scripts/spur-cargo bench -p spur-graph --bench parquet -- bench_end_to_end_mcp_latency_session --noplot`
- Overlay construction: `scripts/spur-cargo bench -p spur-graph --bench overlay -- bench_overlay_construction --noplot`
- Warm base/overlay session: `SPUR_GRAPH_PERF_FIXTURE=<artifact.parquet> scripts/spur-cargo bench -p spur-graph --bench overlay -- 'bench_overlay_vs_direct_parquet/(parquet_cached_session|overlay_one_file_session)' --noplot`
- Reuse behavior: `scripts/spur-cargo test -p spur-graph overlay_delta_cache_` and `scripts/spur-cargo test -p spur-analyst write_delta_for_session_reuses_complete_dir_for_the_same_key`
- Incremental: `SPUR_GRAPH_BENCH_FILES=10000 SPUR_GRAPH_BENCH_CHANGE_SET=100 SPUR_GRAPH_BENCH_DIRTY_MODS=10 scripts/spur-cargo bench -p spur-graph --bench incremental -- incremental --quick --noplot`
- Storage: `du -sk .git/spur-graph/artifacts .spur/graph .spur/analyst.duckdb .spur/analyst-overlays`
- Architecture anchors: `crates/spur-graph/ARCHITECTURE.md`, `crates/spur-graph/src/query_client.rs`, `crates/spur-graph/src/store/cache.rs`, `crates/spur-analyst/src/overlay/mod.rs`, and `crates/spur-analyst/src/db/paths.rs`.

### Interpretation limits

Core and incremental Criterion measurements ran on the configured remote VM; the new warm base/overlay comparison ran locally against the live artifact. Live MCP timing used this agent/tool path with 15 samples after 3 warmups. Storage is a point-in-time measurement. The 2 GiB pool and 1 MiB per-overlay cap are explicit comparison assumptions. Solver results prove only the encoded finite rules.